In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [14]:
Xtr[120000]

tensor([1, 1, 4])

In [9]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [23]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

#loss = -logprobs[(counts * counts_sum_inv).log()].mean()


# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3408, grad_fn=<NegBackward0>)

In [298]:
'''
Differntiation Eq: loss = -logprobs[range(n), Yb].mean()

logprobs --> 32,27
dlogprobs also has to be --> 32,27

loss = -logprobs[range(n), Yb].mean()

ex:

l = -1 * [a b c].mean()
l = -1 * (a+b+c)/3

dl/da = -1/3
dl/db = -1/3
dl/dc = -1/3


Minimize the loss (-logprob[0,8]) needs to be close to zero 
(intuitively for this input record we want the model to predict high probability for the index 8
for correct prediction i.e if prob[0,8] -> 1 , -logprob[0,8] -> 0 
for wrong prediction i.e if prob[0,8] -> 0 , -logprob[0,8] -> inf (high numbers) 
If model learns correctly the loss will be less and if not learned loss is high
 )

One example in the mini batch

-logprobs[0] Yb[0] = 8

tensor([2.6489, 2.3769, 4.0093, 3.0809, 3.9804, 2.4573, 3.7142, 3.3466, 4.0572,
        3.4673, 3.2612, 3.2994, 3.3732, 3.5206, 3.4410, 4.3076, 4.7607, 3.9898,
        4.1223, 2.8900, 3.0320, 3.8568, 3.6767, 2.6381, 2.7981, 3.5875, 3.8281],
       grad_fn=<NegBackward0>)

dlogprobs[0]

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -0.0312,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000])

Intuitively the 8th indices has a grad of -0.0312 which is trying to minimize the -logprob[0,8] towards 0 since it is having a high value of 4.0572


dlogprobs[range(n), Yb] = -1.0/n 

'''
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] =  -1.0 / n
cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


In [299]:
'''

Differntiation Eq: logprobs = probs.log()

probs -> 32,27 (input record of a batch, this tensor holds the probability distribution from the model of possible 27 chars to predict the next char)
dprobs -> 32,27 (grads to flow through to adjust the prob which inturn minizes the loss)

loss = -logprobs[range(n),Yb].mean()
logprobs = probs.log() -> (log(probs))

probs -> log() -> logprobs -> -[range(n),Yb].mean() -> loss

dprobs = (dlogprobs/dprobs) * dlogprobs
dprobs = (1.0/probs) * dlogprobs

probs[0] - probability distribution from the model for the next possible char from the vocabulary

tensor([0.0707, 0.0928, 0.0181, 0.0459, 0.0187, 0.0857, 0.0244, 0.0352, 0.0173,
        0.0312, 0.0383, 0.0369, 0.0343, 0.0296, 0.0320, 0.0135, 0.0086, 0.0185,
        0.0162, 0.0556, 0.0482, 0.0211, 0.0253, 0.0715, 0.0609, 0.0277, 0.0218],
       grad_fn=<SelectBackward0>)

dprobs[0] - dprobs[0,8] is non zero since that's the correct output for this input record and the probs[0,8] has to be high
            whereas in this case it's just - 0.0173

            dprobs[0,8] will propagate back into the network and adjust the trainable variables (i.e parameters) to make sure that 
            probs[0,8] ends up with a high value

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -1.8067,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000], grad_fn=<SelectBackward0>)

'''

dprobs = (1/probs) * dlogprobs 
cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [300]:
'''
Differntiation Eq: probs = counts * counts_sum_inv

counts => (4,3)
[[a1 a2 a3]
[b1 b2 b3]
[c1 c2 c3]
[d1 d2 d3]]

counts_sum_inv => (4,1)
[[i1],
[i2],
[i3],
[i4]]

probs = counts * counts_sum_inv

probs => (4,3) (counts_sum_inv is broadcasted across the row)

[[a1i1 a2i1 a3i1]
[b1i2 b2i2 b3i2]
[c1i3 c2i3 c3i1]
[d1i4 d2i4 d3i4]]

dprobs/dcounts_sum_inv = [[a1 a2 a3]   -> i1 derivatives
                        [b1 b2 b3]    -> i2 derivates
                        [c1 c2 c3]    -> i3 derivates
                        [d1 d2 d3]]   -> i4 derivates

dprobs/dcounts = [[i1 i1 i1]
                [i2 i2 i2]
                [i3 i3 i3]
                [i4 i4 i4]]   Broadcasted version of counts_sum_inv

dcounts_sum_inv = (counts * dprobs).sum(1,keepdim=True)
dcounts = counts_sum_inv * dprobs

Intuition:

Equation takes exponentiated logit outputs (counts) and divides by the inverse of the sum of all
the logit outputs in a row to identify probability distribution of the all the possible 27 characters

counts[0],counts_sum_inv[0]

(tensor([0.7619, 1.0000, 0.1955, 0.4946, 0.2012, 0.9228, 0.2626, 0.3792, 0.1863,
         0.3361, 0.4130, 0.3975, 0.3693, 0.3187, 0.3450, 0.1450, 0.0922, 0.1993,
         0.1746, 0.5987, 0.5194, 0.2277, 0.2726, 0.7701, 0.6563, 0.2980, 0.2343],
        grad_fn=<SelectBackward0>),
 tensor([0.0928], grad_fn=<SelectBackward0>)

dcounts[0],dcounts_sum_inv[0]

(tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         -0.1677,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000], grad_fn=<SelectBackward0>),
 tensor([-0.3366], grad_fn=<SelectBackward0>)

'''
dcounts = counts_sum_inv*dprobs
dcounts_sum_inv = (counts * dprobs).sum(1,keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)


counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


In [301]:
'''
Differntiation Eq: counts_sum_inv = counts_sum**-1 

y = x ** n

dy/dx = n * x ** (n-1)

counts_sum -> (32,1)  Sum of all the exponentiated logits 
counts_sum_inv -> (32,1) Inverse of counts_sum

dcounts_sum = -1.0 * counts_sum ** -2 * dcounts_sum_inv

'''

dcounts_sum =  -1.0 * (counts_sum ** -2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


In [302]:
'''
Differntiation Eq: counts_sum = counts.sum(1, keepdims=True)

counts = [ [a1 a2 a3]
           [b1 b2 b3]
           [c1 c2 c3]]

counts_sum = counts.sum(1,keepdims=True) = [[a1+a2+a3]
                                            [b1+b2+b3]
                                            [c1+c2+c3]]

dcounts += torch.ones_like(counts) * dcounts_sum

Intuition for local derivative:

Y = x1+x2+x3

dx1 = 1
dx2 = 1
dx3 = 1

dcounts[0]

tensor([ 0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,
        -0.1648,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,
         0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,  0.0029,
         0.0029,  0.0029,  0.0029], grad_fn=<SelectBackward0>)

'''

dcounts += torch.ones_like(counts) *  dcounts_sum  
cmp('counts', dcounts, counts)

counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [303]:
'''
Differentiation Eq : counts = norm_logits.exp()

norm_logits.shape -> 32,27
dcounts.shape -> 32,27

dnorm_logits.shape -> 32,27

dnorm_logits = norm_logits * dcounts

dnorm_logits[0]

tensor([ 0.0022,  0.0029,  0.0006,  0.0014,  0.0006,  0.0027,  0.0008,  0.0011,
        -0.0307,  0.0010,  0.0012,  0.0012,  0.0011,  0.0009,  0.0010,  0.0004,
         0.0003,  0.0006,  0.0005,  0.0017,  0.0015,  0.0007,  0.0008,  0.0022,
         0.0019,  0.0009,  0.0007], grad_fn=<SelectBackward0>)

'''

dnorm_logits = norm_logits.exp() * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0


In [199]:
logits.shape

torch.Size([32, 27])

In [203]:
logit_maxes.shape

torch.Size([32, 1])

In [304]:
'''
Differentiation eq : norm_logits = logits - logit_maxes 

Y = x1 - x2
dx1 = 1
dx2 = -1

logits = [[a1 a2 a3]
          [b1 b2 b3]
          [c1 c2 c3]]

logit_maxes = [[a2]
                [b1]
                [c3]]  ---> max values in each tensor

broadcasted logit_maxes

[[a2 a2 a2]
[b1 b1 b1]
[c3 c3 c3]]

norm_logits = [[a1-a2 a2-a2 a3-a2]
                [b1-b1 b2-b1 b3-b1]
                [c1-c3 c2-c3 c3-c3]]


local derivative 

[[-1 -1 -1]
[-1 -1 -1]
[-1 -1 -1]]  * dnorm_logits

[[-dn_l00 -dn_l01 -dn_l02]
[-dn_l10 -dn_l11 -dn_l12]
[-dn_l20 -dn_l21 -dn_l22]] 

Since the single variable was broadcasted across row sum up along the row to get each variables gradient



Let's take the first element

a1 - (a2+h) - (a1 -a2)/h = -h/h = -1

norm_logits -> 32,27
logits -> 32,27
logit_maxes -> 32,1

dlogits = 1 * dnorm_logits

dlogit_maxes = ( -1 * dnorm_logits ).sum(1,keepdim=True)

'''
dlogits = 1 * dnorm_logits
dlogit_maxes = ( -1 * dnorm_logits ).sum(1,keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)


logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0


In [305]:
'''
Differentiation Eq:logit_maxes = logits.max(1, keepdim=True).values

logit_maxes -> 32,1
logits -> 32,27

Y = max [x1 x2 x3 x4]
Let's assume x3 is maximum

Y = x3

dx1 =dx2 = dx4= 0
dx3 = 1

logits -> [[a1 a2 a3]
           [b1 b2 b3]
           [c1 c2 c3]]

logits.max -> [[a2]
               [b1]
               [c3]]

Only elements a2,b1,c3 will carry the out grad to counts and rest all are 0 because of max function

dlogits[0]

tensor([ 0.0022,  0.0029,  0.0006,  0.0014,  0.0006,  0.0027,  0.0008,  0.0011,
        -0.0307,  0.0010,  0.0012,  0.0012,  0.0011,  0.0009,  0.0010,  0.0004,
         0.0003,  0.0006,  0.0005,  0.0017,  0.0015,  0.0007,  0.0008,  0.0022,
         0.0019,  0.0009,  0.0007], grad_fn=<SelectBackward0>)
'''
mask = torch.zeros_like(logits)
mask[torch.arange(logits.size(0)), logits.argmax(dim=1)] = 1

dlogits += mask * dlogit_maxes
cmp('logits', dlogits, logits)


logits          | exact: True  | approximate: True  | maxdiff: 0.0


In [217]:
logits.shape

torch.Size([32, 27])

In [ ]:
'''
Differentiation Eq: logits = h @ W2 + b2 

logits -> (32,27)

h -> (32, 64)
W2 -> (64,27)
b2 -> (27)

Previous Layer has 64 logits hence 64 columns for each input vector in the batch (32) for h
W2 -> the next layer has 27 output logits hence mapping all the 64 logits for each output logit in the current layer (27)
b2 -> 27 bias tensors

h @ W2 -> 32,27 -> +b2 (the 27 bias tensor are added to each of the 27 output logit is added to every input tensors in the batch along the row)


dh = W2 @ dlogits  64,27 @ 32,27   
dW2 = h @ dlogits  32,64 @ 32,27
db2 = 1 @ dlogits  1 @ 32, 27

2,3

batch size of 2 
previous layer nodes = 3
next layer nodes = 4

(2,3)
h = [[h0 h1 h2] => first input tensor 3 logit outputs from the prev layer
    [y0 y1 y2] => second input tensor 3 logit outputs from the prev layer

(3,4) 
W = [[w00 w10 w20 w30] => 1st element weights of all the 4 nodes in the next layer
    [w01 w11 w21,w31] =>  2nd element weights of all the 4 nodes in the next layer
    [w02 w12 w22 w32]] => 3rd element weights of all the 4 nodes in the next layer


h @ W => 2,3 @ 3,4 = 2,4 (next layer outputs for two input tensors (batch size = 2))

(2,4)
h@W

[[h0w00+h1w01+h2+w02 h0w10+h1w11+h2+w12 h0w20+h1w21+h2+w22 h0w30+h1w31+h2+w32]
[y0w00+y1w01+y2w02 y0w10+y1w11+y2w12 y0w20+y1w21+y2w22 y0w30+y1w31+y2w32]] 

To add the bias needs to be broadcasted for both the input tensors 
(4,) => (1,4)
b = [b1 b2 b3 b4]

b= [[b1 b2 b3 b4] 
    [b1 b2 b3 b4]]


[[h0w00+h1w01+h2+w02+b1 h0w10+h1w11+h2+w12+b2 h0w20+h1w21+h2+w22+b3 h0w30+h1w31+h2+w32+b4]
[y0w00+y1w01+y2w02+b1 y0w10+y1w11+y2w12+b2 y0w20+y1w21+y2w22+b3 y0w30+y1w31+y2w32+b4]] 

b1 => b1+h



2h/h = 2

adding bias

lets find db1

logits = h @ W2 + b

db = 1 * dlogits (32,27)
'''

In [240]:
b2.grad

tensor([-0.1199, -0.0588,  0.0352, -0.0248,  0.0319, -0.0191,  0.0430,  0.0113,
        -0.0251, -0.0906,  0.0371, -0.0003,  0.0071,  0.0351, -0.0636,  0.0146,
         0.0355,  0.0310, -0.0273,  0.0140,  0.0076,  0.0302,  0.0131,  0.0368,
         0.0413,  0.0032,  0.0015])

In [246]:
dlogits.sum(0,keepdim=True).view(27)

tensor([-0.1199, -0.0588,  0.0352, -0.0248,  0.0319, -0.0191,  0.0430,  0.0113,
        -0.0251, -0.0906,  0.0371, -0.0003,  0.0071,  0.0351, -0.0636,  0.0146,
         0.0355,  0.0310, -0.0273,  0.0140,  0.0076,  0.0302,  0.0131,  0.0368,
         0.0413,  0.0032,  0.0015], grad_fn=<ViewBackward0>)

In [247]:
#(W2 @ dlogits).shape

dh = dlogits @ torch.transpose(W2, 0, 1)
dW2 = torch.transpose(h,0,1) @ dlogits
db2 = dlogits.sum(0,keepdim=True).view(27)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)


h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0


In [249]:
hpreact.shape

h.shape

torch.Size([32, 64])

In [255]:
dhpreact.shape

torch.Size([32, 64])

In [306]:
'''
Differentiation Eq : h = torch.tanh(hpreact)

y= tanh(x)

dx = 1- y**2

'''
dhpreact = (1 - h ** 2) * dh
cmp('hpreact', dhpreact, hpreact)

hpreact         | exact: True  | approximate: True  | maxdiff: 0.0


In [274]:
dhpreact.shape

torch.Size([32, 64])

In [307]:
'''
Differentiation Eq : hpreact = bngain * bnraw + bnbias

hpreact -> 32,64


bngain -> 1,64 
[[bng0 bng2 ......bng63]] Linear layer after layer 1 

bnraw -> 32,64

[[r00 r01 r02 .....r63]    Output of input tensor 1 layer 1 (layer 1 has 64 nodes)
[r10 r11 ....]             Output of input tensor 2 
...
[r320 r321 .......r3263]]  Output of input tensor 32(batch size)

bnbias -> 1,64
[[bnb0 bnb1 bnb2 ......bnb63]]

Layer 1 has 64 nodes hpreact is the linear layer after the batch norm 

dbngain = (bnraw (32,64) * dhpreact (32,64)).sum(0,keepdim=True)
dbnraw = bngain (1,64) * dhpreact (32,64)
'''

dbngain = (bnraw * dhpreact).sum(0,keepdim=True)
cmp('bngain', dbngain, bngain)

dbnraw = (bngain  * dhpreact)
cmp('bnraw', dbnraw, bnraw)


dbnbias = (1 * dhpreact).sum(0,keepdim=True)
cmp('bnbias', dbnbias, bnbias)


bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0


In [276]:
bnraw.shape,bndiff.shape,bnvar_inv.shape

(torch.Size([32, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [308]:
'''
Differentiation Eq: bnraw (32,64) = bndiff (32,64) * bnvar_inv (1,64)

dbndiff = bnvar_inv * dbnraw
dbnvar_inv (1,64 )= (bndiff * dbnraw).sum(0,keepdim=True)

'''
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0,keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)


bnvar_inv       | exact: True  | approximate: True  | maxdiff: 0.0


In [286]:
bnvar_inv.shape,bnvar.shape,bnvar_inv.shape

(torch.Size([1, 64]), torch.Size([1, 64]), torch.Size([1, 64]))

In [309]:
'''
Differentiation Eq: bnvar_inv = (bnvar + 1e-5)**-0.5



dbnvar = -0.5 * (bnvar +1e-5)**-1.5
'''
dbnvar = (-0.5 * (bnvar +1e-5)**-1.5) * dbnvar_inv
cmp('bnvar', dbnvar, bnvar)

bnvar           | exact: True  | approximate: True  | maxdiff: 0.0


In [290]:
bnvar.shape,bndiff2.shape

(torch.Size([1, 64]), torch.Size([32, 64]))

In [310]:
'''
Differentiation Eq: bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) 

a (2,4) = [[a1 a2 a3 a4]
        [b1 b2 b3 b4]]

y= 1/(n-1) * (a).sum(0,keepdim=True)

let 1/(n-1) = c

y= [[a1*c a2*c a3*c a4*c]
    [b1*c b2*c b3*c b4*c]].sum(0)

y(1,4)= [[a1*c+b1*c a2*c+b2*c a3*c+b3*c a4*c+b4*c]

dbndiff2 = 1/(n-1) * dbnvar

'''
dbndiff2 = 1/(n-1) * dbnvar
cmp('bndiff2', dbndiff2, bndiff2)


bndiff2         | exact: True  | approximate: True  | maxdiff: 0.0


In [295]:
bndiff2.shape,bndiff.shape,dbndiff.shape

(torch.Size([32, 64]), torch.Size([32, 64]), torch.Size([32, 64]))

In [311]:
'''
Differentiation Eq : bndiff2 = bndiff**2

y = x**2
dx = 2 * x ** 1
'''

dbndiff += 2 * (bndiff ** 1) * dbndiff2
cmp('bndiff', dbndiff, bndiff)


bndiff          | exact: True  | approximate: True  | maxdiff: 0.0


In [313]:
bndiff.shape,hprebn.shape,bnmeani.shape

(torch.Size([32, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [316]:
'''
Differentiation Eq: bndiff = hprebn - bnmeani

bndiff (32,64)

hprebn (32,64)

bnmeani (1,64) -> mean of all the 64 logits over the entire batch of inputs (32)


dhprebn = 1 * dbndiff

dbnmeani =  (-1 * dbndiff (32,64)).sum(0,keepdim=True)

'''

dhprebn = 1 * dbndiff
dbnmeani =  (-1 * dbndiff).sum(0,keepdim=True)

cmp('bnmeani', dbnmeani, bnmeani)

bnmeani         | exact: True  | approximate: True  | maxdiff: 0.0


In [319]:
bnmeani.shape, hprebn.shape,dhprebn.shape,dbnmeani.shape

(torch.Size([1, 64]),
 torch.Size([32, 64]),
 torch.Size([32, 64]),
 torch.Size([1, 64]))

In [ ]:
'''
Differentiation Eq: bnmeani = 1/n*hprebn.sum(0, keepdim=True)


y = 1/n * x.sum(0,keepdim=True)

x (2,4)= [[a1 a2 a3 a4]
         [b1 b2 b3 b4]] 

y = 1/n * [[a1+b1 a2+b2 a3+b3 a4+b4]]

da1 = 1/n * 1
da2 = 1/n * 1
da3 = 1/n * 1
...
db4 = 1/n * 1

dx = [[1/n * 1 1/n * 1 1/n * 1 1/n * 1]
     [1/n * 1 1/n * 1 1/n * 1 1/n * 1]] 

dhprebn += 1/n * torch.ones_like(dhprebn) * dbnmeani
'''

In [322]:
dhprebn +=  (1/n * torch.ones_like(dhprebn) * dbnmeani)

cmp('hprebn',dhprebn,hprebn)

hprebn          | exact: True  | approximate: True  | maxdiff: 0.0


In [323]:
hprebn.shape,embcat.shape,W1.shape,b1.shape

(torch.Size([32, 64]),
 torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([64]))

In [ ]:
'''

Differentiation Eq: hprebn (32,64) = embcat (32,30) @ W1 (30,64) + b1 (64)

dembcat = dhprebn (32,64) @ W1 (64,30) 

dW1 = embcat (32,30) @ dhprebn(32,64)

b1 (64) -> (1,64) -> (32,64)  broadcasted into all the batch input tensors and added

db1 = 1 * dhprebn(32,64)

'''

dembcat = dhprebn @ torch.transpose(W1,0,1) 
cmp('embcat', dembcat, embcat)


embcat          | exact: True  | approximate: True  | maxdiff: 0.0


In [325]:
dW1 = torch.transpose(embcat,0,1) @ dhprebn
cmp('W1', dW1, W1)


W1              | exact: True  | approximate: True  | maxdiff: 0.0


In [328]:
dhprebn.sum(0,keepdim=True).view(-1).shape

torch.Size([64])

In [329]:
b1.shape

torch.Size([64])

In [330]:
db1 = dhprebn.sum(0,keepdim=True).view(-1)
cmp('b1', db1, b1)

b1              | exact: True  | approximate: True  | maxdiff: 0.0


In [332]:
emb.shape,embcat.shape,dembcat.shape

(torch.Size([32, 3, 10]), torch.Size([32, 30]), torch.Size([32, 30]))

In [ ]:
'''
Differentiation Eq: embcat = emb.view(emb.shape[0], -1)

No differentiation required since this step just reshapes the embedding tensors from 32,3,10 to 32,30

The gradients need to be just reshaped from 32,30 to 32,3,10
'''

demb = dembcat.reshape(emb.shape[0],emb.shape[1],emb.shape[2])
cmp('emb', demb, emb)

emb             | exact: True  | approximate: True  | maxdiff: 0.0


In [339]:
demb.shape,C.shape,Xb.shape

(torch.Size([32, 3, 10]), torch.Size([27, 10]), torch.Size([32, 3]))

In [ ]:
'''
Differentiation Eq : emb = C[Xb]

'''


dC = torch.zeros_like(C)
for x,y in zip(demb,Xb):
    for i in range(block_size): # Context 
        dC[y[i].item()] += x[i]
cmp('C', dC, C)


C               | exact: True  | approximate: True  | maxdiff: 0.0
